In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# Configuration
NUM_USERS = 1000
START_DATE = datetime(2026, 1, 1)

print("Starting Apple Services Synthetic Data Pipeline...")

# =============================================================================
# 1. GENERATE DIMENSION TABLES
# =============================================================================

# -- Dim App --
dim_app = pd.DataFrame({
    'app_id': [101, 102, 103, 104, 105],
    'app_name': ['Apple Music', 'Apple TV+', 'iCloud+', 'Apple Arcade', 'Apple Fitness+'],
    'monthly_price': [10.99, 9.99, 2.99, 6.99, 9.99],
    'trial_duration_days': [90, 7, 30, 30, 30]
})

# -- Dim Campaign --
dim_campaign = pd.DataFrame({
    'campaign_id': [201, 202, 203, 204],
    'campaign_name': [
        'Holiday Bundle Promo',
        'iPhone Upgrade Cross-Sell',
        'TikTok Influencer Campaign',
        'Organic Search'
    ],
    'channel': ['Email', 'In-App Notification', 'Paid Social', 'Organic']
})

# -- Dim User --
user_ids = [10000 + i for i in range(NUM_USERS)]
dim_user = pd.DataFrame({
    'user_id': user_ids,
    'region': np.random.choice(['US', 'EU', 'APAC', 'LATAM'], size=NUM_USERS, p=[0.50, 0.25, 0.15, 0.10]),
    'signup_date': [START_DATE + timedelta(days=int(np.random.randint(0, 60))) for _ in range(NUM_USERS)]
})

# =============================================================================
# 2. GENERATE FACT TABLE (Subscriptions)
# =============================================================================
fact_records = []

app_ids = dim_app['app_id'].values
campaign_ids = dim_campaign['campaign_id'].values
campaign_probs = [0.20, 0.40, 0.25, 0.15] # Weighting for organic vs paid

for _, user in dim_user.iterrows():
    # Simulate that a user tries 1 to 3 different apps over time
    num_subs = np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])
    chosen_apps = np.random.choice(app_ids, size=num_subs, replace=False)

    for app_id in chosen_apps:
        campaign_id = np.random.choice(campaign_ids, p=campaign_probs)

        # Trial start date must be on or after user signup date
        trial_start = user['signup_date'] + timedelta(days=int(np.random.randint(0, 30)))
        trial_end = trial_start + timedelta(days=30)

        # Business Logic: Specific campaigns convert better
        base_conv_prob = 0.35
        if campaign_id == 202: base_conv_prob = 0.55   # High intent (Cross-sell)
        elif campaign_id == 203: base_conv_prob = 0.20 # Low intent (TikTok)
        elif campaign_id == 204: base_conv_prob = 0.45 # Medium-High (Organic)

        converted = np.random.rand() < base_conv_prob

        if converted:
            status = 'Active' if np.random.rand() < 0.75 else 'Canceled'
            # If they converted, they stayed past trial, generating at least some revenue
            num_months_paid = np.random.randint(1, 5) if status == 'Canceled' else 5
            monthly_price = dim_app[dim_app['app_id'] == app_id]['monthly_price'].values[0]
            total_revenue = round(num_months_paid * monthly_price, 2)
        else:
            status = 'Trial_Expired'
            total_revenue = 0.0

        fact_records.append({
            'user_id': user['user_id'],
            'app_id': app_id,
            'campaign_id': campaign_id,
            'trial_start_date': trial_start.strftime('%Y-%m-%d'),
            'trial_end_date': trial_end.strftime('%Y-%m-%d'),
            'status': status,
            'total_revenue_usd': total_revenue
        })

fact_subscriptions = pd.DataFrame(fact_records)

# =============================================================================
# 3. EXPORT ALL TO CSV
# =============================================================================
dim_app.to_csv('dim_app.csv', index=False)
dim_campaign.to_csv('dim_campaign.csv', index=False)
dim_user.to_csv('dim_user.csv', index=False)
fact_subscriptions.to_csv('fact_subscriptions.csv', index=False)

print("Pipeline Complete! 4 pristine CSV files generated:")
print(f" - dim_app.csv ({len(dim_app)} rows)")
print(f" - dim_campaign.csv ({len(dim_campaign)} rows)")
print(f" - dim_user.csv ({len(dim_user)} rows)")
print(f" - fact_subscriptions.csv ({len(fact_subscriptions)} rows)")

Starting Apple Services Synthetic Data Pipeline...
Pipeline Complete! 4 pristine CSV files generated:
 - dim_app.csv (5 rows)
 - dim_campaign.csv (4 rows)
 - dim_user.csv (1000 rows)
 - fact_subscriptions.csv (1497 rows)
